In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q roboflow

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git@main'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
import json
import os

dataset_path = "data/processed/object_detection/faster_rcnn_coco"

for split in ["train", "valid"]:
    with open(f"{dataset_path}/{split}/_annotations.coco.json") as f:
        data = json.load(f)

    # Rimappa category_id da 0 a 9
    for ann in data["annotations"]:
        ann["category_id"] = ann["category_id"] % 10

    # Rimuovi bbox non valide (larghezza/altezza <= 0)
    valid_annotations = []
    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        if w > 0 and h > 0:
            valid_annotations.append(ann)
        else:
            print(f"Rimosso bbox non valida in {split}:", ann)
    data["annotations"] = valid_annotations

    # Mantieni solo 10 categorie
    unique_cats = {cat["id"]: cat for cat in data["categories"]}.values()
    data["categories"] = list(unique_cats)[:10]

    # Salva JSON corretto
    with open(f"{dataset_path}/{split}/_annotations.coco.json", "w") as f:
        json.dump(data, f)

print("Dataset pulito e rimappato correttamente!")


In [ ]:
for split in ["train","valid"]:
    with open(f"{dataset_path}/{split}/_annotations.coco.json") as f:
        data = json.load(f)

    cat_ids = [ann["category_id"] for ann in data["annotations"]]
    print(f"{split} category_id:", set(cat_ids))

    invalid_bbox = [ann for ann in data["annotations"] if ann["bbox"][2] <= 0 or ann["bbox"][3] <= 0]
    print(f"{split} bbox non valide trovate:", len(invalid_bbox))

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog

# rimuove  registrazioni vecchie
for d in ["chairs_train", "chairs_val"]:
    if d in DatasetCatalog.list():
        DatasetCatalog.remove(d)
    if d in MetadataCatalog.list():
        MetadataCatalog.remove(d)

In [ ]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "chairs_train", {},
    f"{dataset_path}/train/_annotations.coco.json",
    f"{dataset_path}/train"
)

register_coco_instances(
    "chairs_val", {},
    f"{dataset_path}/valid/_annotations.coco.json",
    f"{dataset_path}/valid"
)

In [ ]:
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("chairs_train",)
cfg.DATASETS.TEST = ("chairs_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 10  # 10 modelli di sedie

cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 10000
cfg.OUTPUT_DIR = "models/faster_rcnn"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [ ]:
from detectron2.engine import DefaultTrainer

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# costruzione evaluator COCO sul validation set
evaluator = COCOEvaluator("chairs_val", cfg, False, output_dir="models/faster_rcnn")
val_loader = build_detection_test_loader(cfg, "chairs_val")

# Calcola tutte le metriche
metrics = inference_on_dataset(trainer.model, val_loader, evaluator)
print("Metriche di validation:")
print(metrics)


In [ ]:
from detectron2.data import MetadataCatalog

metadata = MetadataCatalog.get("chairs_val")
bbox_metrics = metrics['bbox']

print("Class-wise simplified Precision/Recall (from AP50 / AR):\n")
for i, name in enumerate(metadata.thing_classes):
    ap50_key = f"AP-{name}"
    if ap50_key in bbox_metrics:
        precision = bbox_metrics[ap50_key] / 100  # AP50 è in %

        recall = bbox_metrics.get(f"AR-{name}", precision) / 100
        f1 = 2*precision*recall / (precision + recall + 1e-6)
        print(f"{name}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")


In [ ]:
print(model)